<a href="https://colab.research.google.com/github/lekhasree15/AI_LAB/blob/main/lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class Clause:
    def __init__(self, literals):
        # A clause is a set of literals, e.g., [('¬food', 'x'), ('likes', 'John', 'x')]
        self.literals = tuple(sorted(literals))

    def __repr__(self):
        if not self.literals:
            return "[] (Contradiction)"
        return " ∨ ".join(f"{'' if lit[0] else '¬'}{lit[1]}({', '.join(lit[2:])})" for lit in self.literals)

    def __eq__(self, other):
        return self.literals == other.literals

    def __hash__(self):
        return hash(self.literals)

# Helper to create positive/negative literals easily
# format: (sign, predicate, *arguments) where sign is True for positive, False for negative
def Pos(pred, *args): return (True, pred, *args)
def Neg(pred, *args): return (False, pred, *args)

# Define the Knowledge Base clauses
kb_clauses = [
    Clause([Neg('food', 'x'), Pos('likes', 'John', 'x')]),              # a
    Clause([Pos('food', 'Apple')]),                                      # b1
    Clause([Pos('food', 'vegetables')]),                                 # b2
    Clause([Neg('eats', 'x', 'y'), Pos('killed', 'x'), Pos('food', 'y')]),# c
    Clause([Pos('eats', 'Anil', 'Peanuts')]),                           # d1
    Clause([Pos('alive', 'Anil')]),                                      # d2
    Clause([Neg('eats', 'Anil', 'x'), Pos('eats', 'Harry', 'x')]),       # e
    Clause([Pos('killed', 'x'), Pos('alive', 'x')]),                     # f
    Clause([Neg('alive', 'x'), Neg('killed', 'x')]),                     # g
]

# Negation of the query: h = likes(John, Peanuts)
negated_query = Clause([Neg('likes', 'John', 'Peanuts')])

print("Initial Knowledge Base Clauses:")
for idx, val in enumerate(kb_clauses, 1):
    print(f"{idx}. {val}")
print(f"Negated Query: {negated_query}\n")


In [8]:
class Clause:
    def __init__(self, literals):
        # A clause is a set of literals, e.g., [('¬food', 'x'), ('likes', 'John', 'x')]
        self.literals = tuple(sorted(literals))

    def __repr__(self):
        if not self.literals:
            return "[] (Contradiction)"
        return " ∨ ".join(f"{'' if lit[0] else '¬'}{lit[1]}({', '.join(lit[2:])})" for lit in self.literals)

    def __eq__(self, other):
        return self.literals == other.literals

    def __hash__(self):
        return hash(self.literals)

# Helper to create positive/negative literals easily
def Pos(pred, *args): return (True, pred, *args)
def Neg(pred, *args): return (False, pred, *args)

# Define the Knowledge Base clauses
kb_clauses = [
    Clause([Neg('food', 'x'), Pos('likes', 'John', 'x')]),              # a
    Clause([Pos('food', 'Apple')]),                                      # b1
    Clause([Pos('food', 'vegetables')]),                                 # b2
    Clause([Neg('eats', 'x', 'y'), Pos('killed', 'x'), Pos('food', 'y')]),# c
    Clause([Pos('eats', 'Anil', 'Peanuts')]),                           # d1
    Clause([Pos('alive', 'Anil')]),                                      # d2
    Clause([Neg('eats', 'Anil', 'x'), Pos('eats', 'Harry', 'x')]),       # e
    Clause([Pos('killed', 'x'), Pos('alive', 'x')]),                     # f
    Clause([Neg('alive', 'x'), Neg('killed', 'x')]),                     # g
]

# Negation of the query: h = likes(John, Peanuts)
negated_query = Clause([Neg('likes', 'John', 'Peanuts')])

print("Initial Knowledge Base Clauses:")
for idx, val in enumerate(kb_clauses, 1):
    print(f"{idx}. {val}")
print(f"Negated Query: {negated_query}\n")

def unify_var(var, val, subst):
    if var in subst:
        return unify(subst[var], val, subst)
    elif val in subst:
        return unify(var, subst[val], subst)
    else:
        subst[var] = val
        return subst

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if subst is False:
        return False
    if x == y:
        return subst
    if isinstance(x, str) and x.islower(): # simple variable check (lowercase strings are variables)
        return unify_var(x, y, subst)
    if isinstance(y, str) and y.islower():
        return unify_var(y, x, subst)
    if isinstance(x, (list, tuple)) and isinstance(y, (list, tuple)) and len(x) == len(y):
        for xi, yi in zip(x, y):
            subst = unify(xi, yi, subst)
        return subst
    return False

def apply_subst(literal, subst):
    sign, pred, *args = literal
    new_args = tuple(subst.get(arg, arg) if (isinstance(arg, str) and arg.islower()) else arg for arg in args)
    # Also handle nested resolution mappings
    new_args = tuple(subst.get(a, a) for a in new_args)
    return (sign, pred, *new_args)

def resolve(clause1, clause2):
    resolvents = []
    for l1 in clause1.literals:
        for l2 in clause2.literals:
            # If they have opposite signs and same predicate
            if l1[0] != l2[0] and l1[1] == l2[1]:
                subst = unify(l1[2:], l2[2:])
                if subst is not False:
                    # Build the new clause
                    new_lits = set()
                    for lit in clause1.literals:
                        if lit != l1:
                            new_lits.add(apply_subst(lit, subst))
                    for lit in clause2.literals:
                        if lit != l2:
                            new_lits.add(apply_subst(lit, subst))
                    resolvents.append((Clause(new_lits), subst))
    return resolvents

# Execution of Step-by-Step Resolution Proof
current_clause = negated_query
proof_steps = [current_clause]

print("Resolution Steps:")
print(f"Step 0 (Negated Query): {current_clause}")

# Chain of resolution to show the direct path to contradiction:
path = [
    (kb_clauses[0], "Resolve with Clause 1: ¬food(x) ∨ likes(John, x)"),
    (kb_clauses[3], "Resolve with Clause 4: ¬eats(x, y) ∨ killed(x) ∨ food(y)"),
    (kb_clauses[4], "Resolve with Clause 5: eats(Anil, Peanuts)"),
    (kb_clauses[8], "Resolve with Clause 9: ¬alive(x) ∨ ¬killed(x)"),
    (kb_clauses[5], "Resolve with Clause 6: alive(Anil)")
]

for i, (kb_c, desc) in enumerate(path, 1):
    resolvents = resolve(current_clause, kb_c)
    if resolvents:
        # Pick the resolved clause
        next_clause, subst = resolvents[0]
        print(f"\n{desc}")
        print(f"Substitution used: {subst}")
        print(f"Step {i}: {next_clause}")
        current_clause = next_clause
    else:
        print(f"Failed to resolve at step {i}")
        break

if len(current_clause.literals) == 0:
    print("\nSuccess! Empty clause [] reached. The query 'likes(John, Peanuts)' is proven to be TRUE.")
else:
    print("\nProof search failed to find a contradiction.")

Initial Knowledge Base Clauses:
1. ¬food(x) ∨ likes(John, x)
2. food(Apple)
3. food(vegetables)
4. ¬eats(x, y) ∨ food(y) ∨ killed(x)
5. eats(Anil, Peanuts)
6. alive(Anil)
7. ¬eats(Anil, x) ∨ eats(Harry, x)
8. alive(x) ∨ killed(x)
9. ¬alive(x) ∨ ¬killed(x)
Negated Query: ¬likes(John, Peanuts)

Resolution Steps:
Step 0 (Negated Query): ¬likes(John, Peanuts)

Resolve with Clause 1: ¬food(x) ∨ likes(John, x)
Substitution used: {'x': 'Peanuts'}
Step 1: ¬food(Peanuts)

Resolve with Clause 4: ¬eats(x, y) ∨ killed(x) ∨ food(y)
Substitution used: {'y': 'Peanuts'}
Step 2: ¬eats(x, Peanuts) ∨ killed(x)

Resolve with Clause 5: eats(Anil, Peanuts)
Substitution used: {'x': 'Anil'}
Step 3: killed(Anil)

Resolve with Clause 9: ¬alive(x) ∨ ¬killed(x)
Substitution used: {'x': 'Anil'}
Step 4: ¬alive(Anil)

Resolve with Clause 6: alive(Anil)
Substitution used: {}
Step 5: [] (Contradiction)

Success! Empty clause [] reached. The query 'likes(John, Peanuts)' is proven to be TRUE.
